In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import poisson
from pathlib import Path
from itertools import combinations
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
PROCESSED  = Path('../data/processed')
MODELS     = Path('../models')
PREDICTIONS = Path('../predictions')

In [ ]:
# 2026 World Cup groups — official draw from Dec 5 2025
GROUPS = {
    'A': ['Mexico', 'South Korea', 'South Africa', 'Czech Republic'],
    'B': ['Canada', 'Switzerland', 'Qatar', 'Bosnia and Herzegovina'],
    'C': ['Brazil', 'Morocco', 'Scotland', 'Haiti'],
    'D': ['United States', 'Australia', 'Paraguay', 'Turkey'],
    'E': ['Germany', 'Ecuador', 'Ivory Coast', 'Curacao'],
    'F': ['Netherlands', 'Japan', 'Tunisia', 'Sweden'],
    'G': ['Belgium', 'Iran', 'Egypt', 'New Zealand'],
    'H': ['Spain', 'Uruguay', 'Saudi Arabia', 'Cape Verde'],
    'I': ['France', 'Senegal', 'Norway', 'Iraq'],
    'J': ['Argentina', 'Austria', 'Algeria', 'Jordan'],
    'K': ['Portugal', 'Colombia', 'Uzbekistan', 'DR Congo'],
    'L': ['England', 'Croatia', 'Panama', 'Ghana'],
}

print(f'{len(GROUPS)} groups, {sum(len(v) for v in GROUPS.values())} teams')

In [ ]:
# load rankings and elo to build feature vectors for WC matches
match_df  = pd.read_csv(PROCESSED / 'match_features.csv', parse_dates=['date'])
rf        = joblib.load(MODELS / 'random_forest.pkl')

# get most recent ranking and points for each team
latest = (match_df.sort_values('date')
          .groupby('home_team')
          .last()[['home_rank', 'home_points', 'home_elo']]
          .rename(columns={'home_rank': 'rank', 'home_points': 'points', 'home_elo': 'elo'})
          .reset_index()
          .rename(columns={'home_team': 'team'}))

# also check away team records for teams that might only appear as away
latest_away = (match_df.sort_values('date')
               .groupby('away_team')
               .last()[['away_rank', 'away_points', 'away_elo']]
               .rename(columns={'away_rank': 'rank', 'away_points': 'points', 'away_elo': 'elo'})
               .reset_index()
               .rename(columns={'away_team': 'team'}))

team_info = pd.concat([latest, latest_away]).groupby('team').last().reset_index()
print(f'team info available for {len(team_info)} teams')
team_info.head()

In [ ]:
# load poisson team strength
team_strength = pd.read_csv(PROCESSED / 'team_strength.csv', index_col=0)
goal_avgs     = pd.read_csv(PROCESSED / 'goal_averages.csv', index_col=0, header=None)
avg_home_goals = float(goal_avgs.loc['avg_home_goals'].values[0])
avg_away_goals = float(goal_avgs.loc['avg_away_goals'].values[0])

def get_team_info(team):
    row = team_info[team_info['team'] == team]
    if len(row) == 0:
        return None, None, None
    return float(row['rank'].values[0]), float(row['points'].values[0]), row['elo'].values[0]

def predict_match_rf(team1, team2):
    r1, p1, e1 = get_team_info(team1)
    r2, p2, e2 = get_team_info(team2)
    if r1 is None or r2 is None:
        return None

    # recent form — last 5 game averages from dataset
    def get_form(team, col_scored, col_conceded, team_col):
        rows = match_df[match_df[team_col] == team].sort_values('date').tail(5)
        if len(rows) == 0:
            return 1.5, 1.0
        return rows[col_scored].mean(), rows[col_conceded].mean()

    gs_h, gc_h = get_form(team1, 'home_score', 'away_score', 'home_team')
    gs_a, gc_a = get_form(team2, 'away_score', 'home_score', 'away_team')

    elo_diff = (e1 - e2) if (pd.notna(e1) and pd.notna(e2)) else 0

    features = pd.DataFrame([{
        'rank_diff':           r1 - r2,
        'point_diff':          p1 - p2,
        'elo_diff':            elo_diff,
        'is_friendly':         0,
        'goals_scored_home':   gs_h,
        'goals_conceded_home': gc_h,
        'goals_scored_away':   gs_a,
        'goals_conceded_away': gc_a,
    }])

    probs = rf.predict_proba(features)[0]
    classes = rf.classes_
    prob_map = dict(zip(classes, probs))
    return prob_map.get(1, 0), prob_map.get(0, 0), prob_map.get(-1, 0)

# test
p = predict_match_rf('Brazil', 'France')
print(f'Brazil vs France — win: {p[0]:.1%}  draw: {p[1]:.1%}  loss: {p[2]:.1%}')

In [ ]:
def simulate_group(group_name, teams, n_simulations=10000):
    results = {t: {'points': 0, 'gd': 0, 'first': 0, 'top2': 0, 'top3': 0} for t in teams}

    for _ in range(n_simulations):
        pts = {t: 0 for t in teams}
        gd  = {t: 0 for t in teams}

        for t1, t2 in combinations(teams, 2):
            p = predict_match_rf(t1, t2)
            if p is None:
                continue
            pw, pd_, pl = p
            r = np.random.choice(['w', 'd', 'l'], p=[pw, pd_, pl])
            # simulate scoreline for goal difference
            g1 = np.random.poisson(1.5)
            g2 = np.random.poisson(1.2)
            if r == 'w':
                g1 = max(g1, g2 + 1)
                pts[t1] += 3
            elif r == 'd':
                g1 = g2
                pts[t1] += 1
                pts[t2] += 1
            else:
                g2 = max(g2, g1 + 1)
                pts[t2] += 3
            gd[t1] += g1 - g2
            gd[t2] += g2 - g1

        # rank by points then goal difference
        standings = sorted(teams, key=lambda t: (pts[t], gd[t]), reverse=True)
        results[standings[0]]['first'] += 1
        for i, t in enumerate(standings[:3]):
            if i < 2:
                results[t]['top2'] += 1
            results[t]['top3'] += 1

    # convert to probabilities
    for t in teams:
        for k in ['first', 'top2', 'top3']:
            results[t][k] /= n_simulations

    return results

# test on one group first
print('simulating Group C (Brazil, Morocco, Scotland, Haiti)...')
gc = simulate_group('C', GROUPS['C'], n_simulations=1000)
for team, r in sorted(gc.items(), key=lambda x: -x[1]['top2']):
    print(f'  {team:20s}  1st: {r["first"]:.1%}  top2: {r["top2"]:.1%}')

In [ ]:
# simulate all 12 groups
print('simulating all groups (this takes ~2 minutes)...')
all_results = {}
for group, teams in GROUPS.items():
    print(f'  group {group}...', end=' ')
    all_results[group] = simulate_group(group, teams, n_simulations=5000)
    print('done')

print('all groups done')

In [ ]:
# build a flat dataframe of all teams and their advancement probabilities
rows = []
for group, results in all_results.items():
    for team, stats in results.items():
        rows.append({'group': group, 'team': team, **stats})

probs_df = pd.DataFrame(rows).sort_values(['group', 'top2'], ascending=[True, False])
probs_df.to_csv(PREDICTIONS / 'group_stage_predictions.csv', index=False)
print(probs_df.to_string(index=False))

In [ ]:
# visualisation — top 2 advancement probability per team, grouped
fig, axes = plt.subplots(3, 4, figsize=(18, 14))
axes = axes.flatten()

colors = ['#1a6496', '#2ecc71', '#e67e22', '#e74c3c']

for idx, (group, ax) in enumerate(zip(sorted(GROUPS.keys()), axes)):
    group_data = probs_df[probs_df['group'] == group].sort_values('top2', ascending=True)
    bars = ax.barh(group_data['team'], group_data['top2'],
                   color=colors[:len(group_data)], edgecolor='white', height=0.6)
    ax.set_xlim(0, 1)
    ax.set_xlabel('prob advance')
    ax.set_title(f'Group {group}', fontweight='bold')
    for bar, val in zip(bars, group_data['top2']):
        ax.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
                f'{val:.0%}', va='center', fontsize=9)

plt.suptitle('2026 World Cup — probability of advancing from group stage', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PREDICTIONS / 'group_stage_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved')

In [ ]:
# predicted group winners — highest first-place probability in each group
predicted_winners = {}
predicted_runners = {}

for group, results in all_results.items():
    sorted_teams = sorted(results.items(), key=lambda x: -x[1]['top2'])
    predicted_winners[group] = sorted_teams[0][0]
    predicted_runners[group] = sorted_teams[1][0]

print('Predicted group winners:')
for g in sorted(predicted_winners):
    w = predicted_winners[g]
    r = predicted_runners[g]
    print(f'  Group {g}: 1st {w:25s}  2nd {r}')

In [ ]:
# tournament win probability — simulate knockout rounds 5000 times
def knockout_match(t1, t2):
    p = predict_match_rf(t1, t2)
    if p is None:
        return np.random.choice([t1, t2])
    pw, pd_, pl = p
    # in knockouts draws go to extra time/pens — roughly 50/50 on the draw
    total = pw + pd_/2
    return t1 if np.random.random() < total else t2

def simulate_tournament(n=5000):
    wins = {t: 0 for group in GROUPS.values() for t in group}

    for _ in range(n):
        # determine who advances from each group
        round_of_32 = []
        for group, results in all_results.items():
            sorted_teams = sorted(results.items(), key=lambda x: -x[1]['top2'])
            round_of_32.append(sorted_teams[0][0])  # 1st
            round_of_32.append(sorted_teams[1][0])  # 2nd

        # also add 8 best third place (simplified: pick teams with highest top3 prob)
        third_place = []
        for group, results in all_results.items():
            sorted_teams = sorted(results.items(), key=lambda x: -x[1]['top2'])
            third_place.append((sorted_teams[2][0], sorted_teams[2][1]['top3']))
        third_place.sort(key=lambda x: -x[1])
        round_of_32 += [t[0] for t in third_place[:8]]

        # run knockout rounds
        remaining = round_of_32
        while len(remaining) > 1:
            next_round = []
            for i in range(0, len(remaining), 2):
                if i + 1 < len(remaining):
                    winner = knockout_match(remaining[i], remaining[i+1])
                    next_round.append(winner)
            remaining = next_round

        if remaining:
            wins[remaining[0]] += 1

    return {t: w/n for t, w in wins.items()}

print('simulating full tournament 5000 times...')
tournament_probs = simulate_tournament(5000)
print('done')

In [ ]:
# final chart — WC winner probability for all teams
win_df = (pd.Series(tournament_probs)
            .sort_values(ascending=False)
            .reset_index())
win_df.columns = ['team', 'win_prob']
win_df = win_df[win_df['win_prob'] > 0.001]  # hide teams with <0.1% chance

# colour by group
group_map = {t: g for g, teams in GROUPS.items() for t in teams}
group_colors = plt.cm.tab20.colors
group_list   = sorted(GROUPS.keys())
color_map    = {g: group_colors[i] for i, g in enumerate(group_list)}
bar_colors   = [color_map[group_map.get(t, 'A')] for t in win_df['team']]

fig, ax = plt.subplots(figsize=(12, max(8, len(win_df)*0.35)))
bars = ax.barh(win_df['team'], win_df['win_prob'], color=bar_colors, edgecolor='white')

for bar, val in zip(bars, win_df['win_prob']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=8)

legend_patches = [mpatches.Patch(color=color_map[g], label=f'Group {g}') for g in group_list]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8, ncol=2)

ax.set_xlabel('probability of winning the World Cup')
ax.set_title('2026 FIFA World Cup — predicted winner probabilities', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PREDICTIONS / 'wc2026_winner_probabilities.png', dpi=150, bbox_inches='tight')
plt.show()

win_df.to_csv(PREDICTIONS / 'wc2026_winner_probabilities.csv', index=False)
print('saved')